# Инициализация Spark и подключение к БД

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import broadcast


spark = SparkSession.builder \
    .appName("ETL_Lab2_Final_Fixed_Integrity") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.2,com.clickhouse:clickhouse-jdbc:0.6.0") \
    .getOrCreate()


pg_url = "jdbc:postgresql://db:5432/pet_store_db"
pg_properties = {"user": "admin", "password": "password", "driver": "org.postgresql.Driver"}
ch_url = "jdbc:clickhouse://clickhouse:8123/reports_db?compress=false&ssl=false"
ch_properties = {"user": "admin", "password": "password", "driver": "com.clickhouse.jdbc.ClickHouseDriver"}

# Загрузка и предобработка исходных данных

In [2]:
raw_df = spark.read.jdbc(pg_url, "mock_data", properties=pg_properties)

for field in raw_df.schema.fields:
    if isinstance(field.dataType, StringType):
        raw_df = raw_df.withColumn(field.name, F.trim(F.col(field.name)))

raw_df = raw_df.cache()

# Заполнение таблиц измерений и таблицы фактов

In [3]:
def write_and_read(df, table_name):
    df.coalesce(1).write.jdbc(pg_url, table_name, mode="append", properties=pg_properties)
    return spark.read.jdbc(pg_url, table_name, properties=pg_properties)


dim_suppliers_df = raw_df.selectExpr(
    "supplier_name", 
    "supplier_contact as contact_name", 
    "supplier_country as country", 
    "supplier_city as city"
).distinct().dropDuplicates(["supplier_name"])
dim_suppliers = write_and_read(dim_suppliers_df, "dim_suppliers")


dim_stores_df = raw_df.selectExpr(
    "store_name", 
    "store_country as country", 
    "store_city as city", 
    "store_state as state"
).distinct().dropDuplicates(["store_name"])
dim_stores = write_and_read(dim_stores_df, "dim_stores")


dim_customers_df = raw_df.selectExpr(
    "customer_email as email", 
    "customer_first_name as first_name", 
    "customer_last_name as last_name", 
    "customer_age as age", 
    "customer_pet_type as pet_type", 
    "customer_country as country", 
    "customer_postal_code as postal_code"
).distinct().dropDuplicates(["email"])
dim_customers = write_and_read(dim_customers_df, "dim_customers")


dim_sellers_df = raw_df.selectExpr(
    "seller_email as email", 
    "seller_first_name as first_name", 
    "seller_last_name as last_name", 
    "seller_country as country", 
    "seller_postal_code as postal_code"
).distinct().dropDuplicates(["email"])
dim_sellers = write_and_read(dim_sellers_df, "dim_sellers")


dim_products_df = raw_df.selectExpr(
    "sale_product_id as original_product_id", 
    "product_name", 
    "product_brand as brand", 
    "product_price as price", 
    "product_category", 
    "pet_category"
).distinct().dropDuplicates(["original_product_id", "price"])
dim_products = write_and_read(dim_products_df, "dim_products")


fact_sales_df = raw_df.alias("m") \
    .join(broadcast(dim_customers).alias("c"), F.col("m.customer_email") == F.col("c.email")) \
    .join(broadcast(dim_sellers).alias("sel"), F.col("m.seller_email") == F.col("sel.email")) \
    .join(broadcast(dim_products).alias("p"), 
          (F.col("m.sale_product_id") == F.col("p.original_product_id")) & 
          (F.col("m.product_price") == F.col("p.price"))) \
    .join(broadcast(dim_stores).alias("st"), F.col("m.store_name") == F.col("st.store_name")) \
    .join(broadcast(dim_suppliers).alias("sup"), F.col("m.supplier_name") == F.col("sup.supplier_name")) \
    .selectExpr(
        "m.sale_date",
        "c.customer_id",
        "sel.seller_id",
        "p.product_id",
        "st.store_id",
        "sup.supplier_id", 
        "m.sale_quantity as quantity",
        "m.sale_total_price as total_price"
    )

fact_sales_df.coalesce(1).write.jdbc(pg_url, "fact_sales", mode="append", properties=pg_properties)

# Формирование отчётов ClickHouse

In [4]:
def save_to_ch(df, table_name):
    df.na.fill("N/A").na.fill(0).coalesce(1).write.jdbc(ch_url, table_name, mode="overwrite", properties=ch_properties)

f = fact_sales_df.alias("f")
p = dim_products.alias("p")
c = dim_customers.alias("c")
st = dim_stores.alias("st")
sel = dim_sellers.alias("sel")
sup = dim_suppliers.alias("sup")

analytics_base = f.join(p, "product_id") \
    .join(c, "customer_id") \
    .join(st, "store_id") \
    .join(sel, "seller_id") \
    .join(sup, "supplier_id")

### 1. Витрина продаж по продуктам Цель: Анализ выручки, количества продаж и популярности продуктов. 
- Топ-10 самых продаваемых продуктов.
- Общая выручка по категориям продуктов.
- Средний рейтинг и количество отзывов для каждого продукта.

In [5]:
save_to_ch(analytics_base.groupBy("p.product_name", "p.brand").agg(F.sum("f.quantity").alias("total_sold")).orderBy(F.desc("total_sold")).limit(10), "v1_1_top_10_selling_products")
save_to_ch(analytics_base.groupBy("p.product_category", "p.pet_category").agg(F.sum("f.total_price").alias("revenue")), "v1_2_revenue_by_category")
save_to_ch(raw_df.groupBy("product_name", "product_brand").agg(F.avg("product_rating").alias("avg_rating"), F.sum("product_reviews").alias("reviews_count")), "v1_3_product_rating_reviews")

### 2. Витрина продаж по клиентам Цель: Анализ покупательского поведения и сегментация клиентов. 
- Топ-10 клиентов с наибольшей общей суммой покупок.
- Распределение клиентов по странам.
- Средний чек для каждого клиента.

In [6]:
save_to_ch(analytics_base.groupBy("c.customer_id", "c.first_name", "c.last_name").agg(F.sum("f.total_price").alias("total_spent")).orderBy(F.desc("total_spent")).limit(10), "v2_1_top_10_customers")
save_to_ch(analytics_base.groupBy("c.country").agg(F.countDistinct("c.customer_id").alias("client_count")), "v2_2_customers_by_country")
save_to_ch(analytics_base.groupBy("c.customer_id").agg(F.avg("f.total_price").alias("avg_check")), "v2_3_customer_avg_check")

### 3. Витрина продаж по времени Цель: Анализ сезонности и трендов продаж.
- Месячные и годовые тренды продаж.
- Сравнение выручки за разные периоды (кварталы).
- Средний размер заказа по месяцам.

In [7]:
v3_base = analytics_base.withColumn("year", F.year("f.sale_date")) \
                        .withColumn("quarter", F.quarter("f.sale_date")) \
                        .withColumn("month", F.month("f.sale_date"))

save_to_ch(v3_base.groupBy("year", "month").agg(F.sum("f.total_price").alias("revenue")).orderBy("year", "month"), "v3_1_monthly_trends")
save_to_ch(v3_base.groupBy("quarter").agg(F.sum("f.total_price").alias("quarterly_revenue")).orderBy("quarter"), "v3_2_quarterly_comparison")
save_to_ch(v3_base.groupBy("month").agg(F.avg("f.total_price").alias("avg_order_size")), "v3_3_avg_order_by_month")

### 4. Витрина продаж по магазинам Цель: Анализ эффективности магазинов.
- Топ-5 магазинов с наибольшей выручкой.
- Распределение продаж по городам и странам.
- Средний чек для каждого магазина.

In [8]:
save_to_ch(analytics_base.groupBy("st.store_name").agg(F.sum("f.total_price").alias("revenue")).orderBy(F.desc("revenue")).limit(5), "v4_1_top_5_stores")
save_to_ch(analytics_base.groupBy("st.country", "st.city").agg(F.sum("f.total_price").alias("revenue")), "v4_2_store_sales_geo")
save_to_ch(analytics_base.groupBy("st.store_name").agg(F.avg("f.total_price").alias("avg_check")), "v4_3_store_avg_check")

### 5. Витрина продаж по поставщикам Цель: Анализ эффективности поставщиков.
- Топ-5 поставщиков с наибольшей выручкой.
- Средняя цена товаров от каждого поставщика.
- Распределение продаж по странам поставщиков.

In [9]:
save_to_ch(analytics_base.groupBy("sup.supplier_name").agg(F.sum("f.total_price").alias("revenue")).orderBy(F.desc("revenue")).limit(5), "v5_1_top_5_suppliers")
save_to_ch(analytics_base.groupBy("sup.supplier_name").agg(F.avg("p.price").alias("avg_unit_price")), "v5_2_supplier_avg_price")
save_to_ch(analytics_base.groupBy("sup.country").agg(F.sum("f.total_price").alias("revenue")), "v5_3_supplier_sales_by_country")

### 6. Витрина качества продукции Цель: Анализ отзывов и рейтингов товаров.
- Продукты с наивысшим и наименьшим рейтингом.
- Корреляция между рейтингом и объемом продаж.
- Продукты с наибольшим количеством отзывов.

In [10]:
v6_data = raw_df.groupBy("product_name", "product_brand").agg(
    F.avg("product_rating").alias("rating"),
    F.sum("product_reviews").alias("reviews"),
    F.sum("sale_total_price").alias("sales_vol")
)

top_bottom_rating = v6_data.orderBy(F.desc("rating")).limit(5).union(v6_data.orderBy(F.asc("rating")).limit(5))

save_to_ch(top_bottom_rating, "v6_1_high_low_rated_products")
save_to_ch(v6_data.select("product_name", "product_brand", "rating", "sales_vol"), "v6_2_rating_vs_sales_vol")
save_to_ch(v6_data.orderBy(F.desc("reviews")).limit(10), "v6_3_most_reviewed_products")